In [ ]:
"""Colab-ready high-VRAM hybrid CNN+attention regressor for the Drive `dataset` folder.

Architecture:
- Raw waveform input from `dataset/sample_*.wav`
- Much wider residual 1D CNN with later downsampling, multi-scale pooling, and squeeze-excite
- Temporal self-attention blocks on deeper feature maps for richer context
- Separate heads for frequency, grouped numeric targets, and categorical targets
- Mixed precision, tf.data streaming, and a larger default batch size to push GPU memory harder

Data flow:
- Input: `dataset/parameters.csv` and matching WAV files generated in Colab
- Output: trained `.keras` model, preprocessing bundle, history CSV/plots, predictions CSV, and `results.json`
"""

import json
import os
from pathlib import Path

try:
    from google.colab import drive
except Exception:
    drive = None

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.layers import (
    MultiHeadAttention,
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    Input,
    LayerNormalization,
    Multiply,
    Reshape,
    SpatialDropout1D,
)
from keras.models import Model
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import mixed_precision

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DRIVE_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Unifesp/fmsynth"),
    Path("/content/drive/MyDrive/fmsynth"),
]
if drive is not None and not os.path.ismount('/content/drive'):
    drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = next((path for path in DRIVE_ROOT_CANDIDATES if path.exists()), DRIVE_ROOT_CANDIDATES[0])
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using drive root: {DRIVE_ROOT}")

BASE_PATH = DRIVE_ROOT / "dataset"
print(f"Using dataset folder: {BASE_PATH}")
OUTPUT_DIR = DRIVE_ROOT / "model_training_big4_fmsynth3_0_5_colab"
MODEL_NAME = "model_training_big4_fmsynth3_0_5_colab"
SEED = 42
TRAIN_FRAC = 0.75
VAL_FRAC_OF_TRAIN = 0.2
BATCH_SIZE = int(os.getenv("TRAIN_BATCH_SIZE", "32"))
PRED_BATCH_SIZE = int(os.getenv("PRED_BATCH_SIZE", "64"))
EPOCHS = int(os.getenv("EPOCHS", "280"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "1.5e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "3e-5"))
SHUFFLE_BUFFER = int(os.getenv("SHUFFLE_BUFFER", "16384"))
AUTOTUNE = tf.data.AUTOTUNE

# If you hit OOM on Colab, lower this first.
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.keras.utils.set_random_seed(SEED)
except Exception:
    pass

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(exc)
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision policy:", mixed_precision.global_policy())
else:
    print("GPU not found; running in float32.")

ALGORITHM_MERGE_MAP = {
    # In fm_synth3, dual_chain is an alias of series2x2_parallel1.
    "dual_chain": "series2x2_parallel1",
}


def to_json_scalar(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


def cat_output_name(column_name: str) -> str:
    return f"cat__{column_name}"


def reg_output_name(group_name: str) -> str:
    return f"{group_name}_head"


def plot_metric(history_df, train_key, val_key, ylabel, filename):
    if train_key not in history_df.columns or val_key not in history_df.columns:
        return

    plt.figure(dpi=400)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.plot(history_df["epoch"], history_df[train_key], label=f"{ylabel} Training")
    plt.plot(history_df["epoch"], history_df[val_key], label=f"{ylabel} Validation")
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / filename, dpi=400)
    plt.close()


def build_numeric_groups(numeric_cols: list[str]) -> dict[str, list[str]]:
    ratio_base = ["ratio1", "ratio2", "ratio3", "ratio4", "ratio5", "ratio_carrier"]
    index_base = ["index_12", "index_23", "index_3c", "index_4c", "index_5c"]
    detune_base = ["detune1", "detune2", "detune3", "detune4", "detune5", "detune_carrier"]

    ratio_cols = [c for c in ratio_base if c in numeric_cols]
    index_cols = [c for c in index_base if c in numeric_cols]
    detune_cols = [c for c in detune_base if c in numeric_cols]

    used = set(ratio_cols + index_cols + detune_cols)

    env_cols = [c for c in numeric_cols if c.startswith("env") and c not in used]
    used.update(env_cols)

    phase_cols = [c for c in numeric_cols if c.startswith("phase") and c not in used]
    used.update(phase_cols)

    other_cols = [c for c in numeric_cols if c not in used]

    return {
        "ratio": ratio_cols,
        "index": index_cols,
        "detune": detune_cols,
        "env": env_cols,
        "phase": phase_cols,
        "other": other_cols,
    }


def categorical_loss_weight(col_name: str) -> float:
    if col_name == "algorithm":
        return 0.85
    if col_name == "style":
        return 0.15
    if col_name.startswith("env") and "curve" in col_name:
        return 0.04
    return 0.05


def numeric_head_loss_weight(head_name: str) -> float:
    return {
        "ratio_head": 3.4,
        "index_head": 2.6,
        "detune_head": 2.0,
        "env_head": 1.0,
        "phase_head": 0.25,
        "other_head": 1.05,
        "freq_head": 3.4,
    }.get(head_name, 1.0)


def numeric_head_hidden_dims(head_name: str) -> tuple[int, int]:
    return {
        "ratio_head": (384, 192),
        "index_head": (320, 160),
        "detune_head": (256, 128),
        "env_head": (448, 224),
        "phase_head": (192, 96),
        "other_head": (256, 128),
        "freq_head": (320, 160),
    }.get(head_name, (128, 64))


def categorical_head_hidden_dim(col_name: str) -> int:
    if col_name == "algorithm":
        return 384
    if col_name == "style":
        return 192
    return 128


# -----------------------------------------------------------------------------
# Load dataset metadata and targets
# -----------------------------------------------------------------------------
meta = {}
meta_path = Path(BASE_PATH) / "meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))

params_path = Path(BASE_PATH) / "parameters.csv"
if not params_path.exists():
    raise FileNotFoundError(
        f"{params_path} not found. Run goolge_colab/gcp_generate_dataset_big4_fmsynth3_0_1.ipynb first and make sure it wrote to {DRIVE_ROOT / 'dataset'}."
    )

# Read targets.
target_raw = pd.read_csv(params_path)
if "algorithm" in target_raw.columns:
    target_raw["algorithm"] = target_raw["algorithm"].replace(ALGORITHM_MERGE_MAP)

tamanho_dataset = int(meta.get("tamanho_dataset", len(target_raw)))
if tamanho_dataset < len(target_raw):
    target_raw = target_raw.iloc[:tamanho_dataset].copy()
    tamanho_dataset = len(target_raw)

sample_rate_out = int(meta.get("sample_rate_out", 16000))
duration_seconds = float(meta.get("duracao_amostras", 4.0))
target_samples = int(round(sample_rate_out * duration_seconds))
print(
    f"Dataset size={tamanho_dataset}, sample_rate={sample_rate_out}, "
    f"duration={duration_seconds}, samples_per_file={target_samples}"
)

if "id" in target_raw.columns:
    sample_ids = target_raw["id"].astype(int).tolist()
else:
    sample_ids = list(range(tamanho_dataset))

audio_paths = np.array([str(Path(BASE_PATH) / f"sample_{sid}.wav") for sid in sample_ids])
missing_paths = [path for path in audio_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing audio file: {missing_paths[0]}")

# Encode categorical and boolean columns.
categorical_maps = {}
target_encoded = target_raw.copy()
for col in target_encoded.columns:
    if target_encoded[col].dtype == object:
        cat = pd.Categorical(target_encoded[col])
        categorical_maps[col] = [str(x) for x in cat.categories]
        target_encoded[col] = cat.codes.astype(np.int32)
    elif target_encoded[col].dtype == bool:
        target_encoded[col] = target_encoded[col].astype(np.int32)

if "id" in target_encoded.columns:
    target_all = target_encoded.drop(columns=["id"]).copy()
else:
    target_all = target_encoded.copy()

# Remove constant targets.
constant_targets = {}
for col in list(target_all.columns):
    if target_all[col].nunique(dropna=False) <= 1:
        constant_targets[col] = to_json_scalar(target_all[col].iloc[0])

target_model = target_all.drop(columns=list(constant_targets.keys())).copy()
if target_model.empty:
    raise ValueError("All targets were removed as constants.")

target_columns = list(target_model.columns)
categorical_cols = [col for col in target_columns if col in categorical_maps]
numeric_cols = [col for col in target_columns if col not in categorical_cols]

# frequency in its own head, log2-normalized.
freq_col = None
if "frequencia_base" in numeric_cols:
    freq_col = "frequencia_base"
    numeric_cols.remove(freq_col)

numeric_groups = build_numeric_groups(numeric_cols)

print("Target columns:", len(target_columns))
print("Categorical columns:", len(categorical_cols))
print("Numeric groups:", {k: len(v) for k, v in numeric_groups.items()})
print("Removed constant targets:", constant_targets)


# -----------------------------------------------------------------------------
# Split train / validation / test
# -----------------------------------------------------------------------------
all_indices = np.arange(len(target_model))
train_idx = target_all.sample(frac=TRAIN_FRAC, random_state=SEED).index.to_numpy()
test_idx = np.setdiff1d(all_indices, train_idx, assume_unique=False)

train_paths = audio_paths[train_idx]
test_paths = audio_paths[test_idx]
train_raw = target_model.iloc[train_idx].reset_index(drop=True)
test_raw = target_model.iloc[test_idx].reset_index(drop=True)
train_all = target_all.iloc[train_idx].reset_index(drop=True)
test_all = target_all.iloc[test_idx].reset_index(drop=True)

val_local_idx = train_raw.sample(frac=VAL_FRAC_OF_TRAIN, random_state=SEED).index.to_numpy()
fit_local_idx = np.setdiff1d(np.arange(len(train_raw)), val_local_idx, assume_unique=False)

fit_paths = train_paths[fit_local_idx]
val_paths = train_paths[val_local_idx]

# Save split artifacts for reproducibility.
np.save(Path(OUTPUT_DIR) / "train_paths.npy", fit_paths)
np.save(Path(OUTPUT_DIR) / "val_paths.npy", val_paths)
np.save(Path(OUTPUT_DIR) / "test_paths.npy", test_paths)
np.save(Path(OUTPUT_DIR) / "y_train_model.npy", train_raw.to_numpy(dtype=np.float32))
np.save(Path(OUTPUT_DIR) / "y_test_model.npy", test_raw.to_numpy(dtype=np.float32))


# -----------------------------------------------------------------------------
# Prepare labels per head
# -----------------------------------------------------------------------------
y_fit: dict[str, np.ndarray] = {}
y_val: dict[str, np.ndarray] = {}
scaler_by_numeric_head: dict[str, StandardScaler] = {}
numeric_head_specs: dict[str, dict] = {}

for group_name, cols in numeric_groups.items():
    if not cols:
        continue

    head_name = reg_output_name(group_name)
    values = train_raw[cols].to_numpy(dtype=np.float32)
    log2_cols: list[str] = []

    if group_name == "ratio":
        if np.any(values <= 0):
            raise ValueError("There are ratio values <= 0; log2 is not applicable.")
        values = np.log2(values)
        log2_cols = list(cols)

    scaler = StandardScaler()
    values_norm = scaler.fit_transform(values).astype(np.float32)

    y_fit[head_name] = values_norm[fit_local_idx]
    y_val[head_name] = values_norm[val_local_idx]

    scaler_by_numeric_head[head_name] = scaler
    numeric_head_specs[head_name] = {
        "group_name": group_name,
        "cols": list(cols),
        "log2_cols": log2_cols,
    }

scaler_freq = None
if freq_col is not None:
    y_freq = train_raw[freq_col].to_numpy(dtype=np.float32)
    if np.any(y_freq <= 0):
        raise ValueError("frequencia_base contains values <= 0; log2 is not applicable.")

    y_freq_log2 = np.log2(y_freq).reshape(-1, 1)
    scaler_freq = StandardScaler()
    y_freq_norm = scaler_freq.fit_transform(y_freq_log2).astype(np.float32)
    y_fit["freq_head"] = y_freq_norm[fit_local_idx]
    y_val["freq_head"] = y_freq_norm[val_local_idx]

cat_dims = {}
for col in categorical_cols:
    head_name = cat_output_name(col)
    y_col = train_raw[col].to_numpy(dtype=np.int32)
    y_fit[head_name] = y_col[fit_local_idx]
    y_val[head_name] = y_col[val_local_idx]
    cat_dims[col] = len(categorical_maps[col])


# -----------------------------------------------------------------------------
# tf.data pipeline
# -----------------------------------------------------------------------------
def load_audio_only(path):
    audio_bytes = tf.io.read_file(path)
    waveform, _sample_rate = tf.audio.decode_wav(
        audio_bytes,
        desired_channels=1,
        desired_samples=target_samples,
    )
    waveform = tf.ensure_shape(waveform, [target_samples, 1])
    return tf.cast(waveform, tf.float32)


def load_example(path, label):
    return load_audio_only(path), label


def make_labeled_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(
            min(len(paths), SHUFFLE_BUFFER),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    ds = ds.map(load_example, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


def make_audio_dataset(paths):
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(load_audio_only, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(PRED_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


train_ds = make_labeled_dataset(fit_paths, y_fit, training=True)
val_ds = make_labeled_dataset(val_paths, y_val, training=False)
predict_ds = make_audio_dataset(test_paths)

for xb, yb in train_ds.take(1):
    print("Batch shapes:", xb.shape, {k: tuple(v.shape.as_list()) for k, v in yb.items()})


# -----------------------------------------------------------------------------
# Model definition
# -----------------------------------------------------------------------------
def residual_block(x, filters, kernel_size, stride=1, dilation_rate=1, dropout=0.0, name="res"):
    shortcut = x

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        dilation_rate=dilation_rate,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv1",
    )(x)
    x = BatchNormalization(name=f"{name}_bn1")(x)
    x = Activation("gelu", name=f"{name}_act1")(x)
    if dropout > 0:
        x = SpatialDropout1D(dropout, name=f"{name}_drop1")(x)

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=1,
        dilation_rate=dilation_rate,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv2",
    )(x)
    x = BatchNormalization(name=f"{name}_bn2")(x)

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = Conv1D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{name}_skip",
        )(shortcut)
        shortcut = BatchNormalization(name=f"{name}_skip_bn")(shortcut)

    x = Add(name=f"{name}_add")([x, shortcut])
    x = Activation("gelu", name=f"{name}_out")(x)
    return x


def squeeze_excite(x, reduction=16, name="se"):
    channels = int(x.shape[-1])
    hidden_units = max(channels // reduction, 16)
    se = GlobalAveragePooling1D(name=f"{name}_gap")(x)
    se = Dense(hidden_units, activation="gelu", name=f"{name}_dense1")(se)
    se = Dense(channels, activation="sigmoid", name=f"{name}_dense2")(se)
    se = Reshape((1, channels), name=f"{name}_reshape")(se)
    return Multiply(name=f"{name}_scale")([x, se])


def transformer_block(x, num_heads, key_dim, ff_dim, dropout=0.0, name="attn"):
    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout,
        name=f"{name}_mha",
    )(x, x)
    x = Add(name=f"{name}_attn_add")([x, attn])
    x = LayerNormalization(name=f"{name}_attn_ln")(x)

    ff = Dense(ff_dim, activation="gelu", name=f"{name}_ff_dense1")(x)
    if dropout > 0:
        ff = Dropout(dropout, name=f"{name}_ff_drop1")(ff)
    ff = Dense(int(x.shape[-1]), activation=None, name=f"{name}_ff_dense2")(ff)
    x = Add(name=f"{name}_ff_add")([x, ff])
    x = LayerNormalization(name=f"{name}_ff_ln")(x)
    return x


def build_model(input_len, numeric_output_dims, categorical_output_dims, has_freq_head):
    inputs = Input(shape=(input_len, 1), name="audio_input")
    x = BatchNormalization(name="input_bn")(inputs)

    # Larger stem and slower compression to keep more temporal detail alive.
    x = Conv1D(
        filters=96,
        kernel_size=9,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv",
    )(x)
    x = BatchNormalization(name="stem_bn")(x)
    x = Activation("gelu", name="stem_act")(x)

    # Stage 1: high-resolution trunk.
    x = residual_block(x, 96, 7, stride=1, dropout=0.02, name="stage1_block1")
    x = residual_block(x, 96, 7, stride=1, dropout=0.02, name="stage1_block2")
    x = residual_block(x, 96, 5, stride=1, dropout=0.02, dilation_rate=2, name="stage1_block3")
    x = residual_block(x, 96, 5, stride=1, dropout=0.02, dilation_rate=4, name="stage1_block4")
    stage1 = x

    # Stage 2.
    x = residual_block(x, 128, 5, stride=2, dropout=0.03, name="stage2_down")
    x = residual_block(x, 128, 5, stride=1, dropout=0.03, name="stage2_block1")
    x = residual_block(x, 128, 5, stride=1, dropout=0.03, name="stage2_block2")
    x = residual_block(x, 128, 3, stride=1, dropout=0.03, dilation_rate=2, name="stage2_block3")
    stage2 = x

    # Stage 3.
    x = residual_block(x, 176, 5, stride=2, dropout=0.04, name="stage3_down")
    x = residual_block(x, 176, 3, stride=1, dropout=0.04, name="stage3_block1")
    x = residual_block(x, 176, 3, stride=1, dropout=0.04, dilation_rate=2, name="stage3_block2")
    x = residual_block(x, 176, 3, stride=1, dropout=0.04, dilation_rate=4, name="stage3_block3")
    stage3 = x

    # Stage 4.
    x = residual_block(x, 240, 3, stride=2, dropout=0.05, name="stage4_down")
    x = residual_block(x, 240, 3, stride=1, dropout=0.05, name="stage4_block1")
    x = residual_block(x, 240, 3, stride=1, dropout=0.05, dilation_rate=2, name="stage4_block2")
    x = residual_block(x, 240, 3, stride=1, dropout=0.05, dilation_rate=4, name="stage4_block3")
    x = squeeze_excite(x, reduction=16, name="stage4_se")
    x = transformer_block(x, num_heads=6, key_dim=48, ff_dim=768, dropout=0.10, name="stage4_attn")
    stage4 = x

    # Stage 5.
    x = residual_block(x, 320, 3, stride=2, dropout=0.06, name="stage5_down")
    x = residual_block(x, 320, 3, stride=1, dropout=0.06, name="stage5_block1")
    x = residual_block(x, 320, 3, stride=1, dropout=0.06, dilation_rate=2, name="stage5_block2")
    x = residual_block(x, 320, 3, stride=1, dropout=0.06, dilation_rate=4, name="stage5_block3")
    x = transformer_block(x, num_heads=8, key_dim=64, ff_dim=1024, dropout=0.12, name="stage5_attn")
    stage5 = x

    # Stage 6.
    x = residual_block(x, 384, 3, stride=2, dropout=0.08, name="stage6_down")
    x = residual_block(x, 384, 3, stride=1, dropout=0.08, name="stage6_block1")
    x = residual_block(x, 384, 3, stride=1, dropout=0.08, dilation_rate=2, name="stage6_block2")
    x = residual_block(x, 384, 3, stride=1, dropout=0.08, dilation_rate=4, name="stage6_block3")
    x = squeeze_excite(x, reduction=16, name="tail_se")
    x = transformer_block(x, num_heads=8, key_dim=64, ff_dim=1280, dropout=0.12, name="stage6_attn")
    stage6 = x

    pooled = []
    for tensor, prefix in [
        (stage1, "s1"),
        (stage2, "s2"),
        (stage3, "s3"),
        (stage4, "s4"),
        (stage5, "s5"),
        (stage6, "s6"),
    ]:
        pooled.append(GlobalAveragePooling1D(name=f"{prefix}_gap")(tensor))
        pooled.append(GlobalMaxPooling1D(name=f"{prefix}_gmp")(tensor))

    x = Concatenate(name="multi_scale_pool")(pooled)
    x = LayerNormalization(name="head_ln")(x)

    # Heavy shared trunk to increase parameter + activation pressure.
    x = Dense(3072, activation="gelu", name="shared_dense_1")(x)
    x = Dropout(0.45, name="shared_drop_1")(x)
    x = Dense(1536, activation="gelu", name="shared_dense_2")(x)
    x = Dropout(0.35, name="shared_drop_2")(x)
    x = Dense(768, activation="gelu", name="shared_dense_3")(x)
    x = Dropout(0.25, name="shared_drop_3")(x)
    x = Dense(384, activation="gelu", name="shared_dense_4")(x)
    x = Dropout(0.15, name="shared_drop_4")(x)

    outputs = {}

    for head_name, output_dim in numeric_output_dims.items():
        hidden_1, hidden_2 = numeric_head_hidden_dims(head_name)
        hidden_3 = max(hidden_2 // 2, 32)
        head = Dense(hidden_1, activation="gelu", use_bias=True, name=f"{head_name}_dense_1")(x)
        head = Dropout(0.20, name=f"{head_name}_drop_1")(head)
        head = Dense(hidden_2, activation="gelu", use_bias=True, name=f"{head_name}_dense_2")(head)
        head = Dropout(0.10, name=f"{head_name}_drop_2")(head)
        head = Dense(hidden_3, activation="gelu", use_bias=True, name=f"{head_name}_dense_3")(head)
        outputs[head_name] = Dense(
            output_dim,
            activation=None,
            use_bias=True,
            dtype="float32",
            name=head_name,
        )(head)

    if has_freq_head:
        hidden_1, hidden_2 = numeric_head_hidden_dims("freq_head")
        hidden_3 = max(hidden_2 // 2, 32)
        head = Dense(hidden_1, activation="gelu", use_bias=True, name="freq_head_dense_1")(x)
        head = Dropout(0.20, name="freq_head_drop_1")(head)
        head = Dense(hidden_2, activation="gelu", use_bias=True, name="freq_head_dense_2")(head)
        head = Dropout(0.10, name="freq_head_drop_2")(head)
        head = Dense(hidden_3, activation="gelu", use_bias=True, name="freq_head_dense_3")(head)
        outputs["freq_head"] = Dense(
            1,
            activation=None,
            use_bias=True,
            dtype="float32",
            name="freq_head",
        )(head)

    for col, n_classes in categorical_output_dims.items():
        head_name = cat_output_name(col)
        hidden = categorical_head_hidden_dim(col)
        head = Dense(hidden, activation="gelu", use_bias=True, name=f"{head_name}_dense_1")(x)
        head = Dropout(0.20, name=f"{head_name}_drop_1")(head)
        head = Dense(max(hidden // 2, 32), activation="gelu", use_bias=True, name=f"{head_name}_dense_2")(head)
        outputs[head_name] = Dense(
            n_classes,
            activation="softmax",
            use_bias=True,
            dtype="float32",
            name=head_name,
        )(head)

    return Model(inputs, outputs, name="high_vram_hybrid_cnn_attention_regressor")

numeric_output_dims = {
    head_name: len(spec["cols"]) for head_name, spec in numeric_head_specs.items()
}

model = build_model(
    input_len=target_samples,
    numeric_output_dims=numeric_output_dims,
    categorical_output_dims=cat_dims,
    has_freq_head=freq_col is not None,
)
model.summary()

try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )
except AttributeError:
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0,
    )

losses = {}
metrics = {}
loss_weights = {}

for head_name in numeric_head_specs:
    losses[head_name] = tf.keras.losses.Huber(delta=0.75)
    metrics[head_name] = ["mae", "mse"]
    loss_weights[head_name] = numeric_head_loss_weight(head_name)

if freq_col is not None:
    losses["freq_head"] = tf.keras.losses.Huber(delta=0.5)
    metrics["freq_head"] = ["mae", "mse"]
    loss_weights["freq_head"] = numeric_head_loss_weight("freq_head")

for col in categorical_cols:
    head_name = cat_output_name(col)
    losses[head_name] = tf.keras.losses.SparseCategoricalCrossentropy()
    metrics[head_name] = ["sparse_categorical_accuracy"]
    loss_weights[head_name] = categorical_loss_weight(col)

model.compile(
    optimizer=optimizer,
    loss=losses,
    metrics=metrics,
    loss_weights=loss_weights,
)

summary_path = Path(OUTPUT_DIR) / "model_summary.txt"
with summary_path.open("w", encoding="utf-8") as fh:
    model.summary(print_fn=lambda line: fh.write(line + "\n"))


# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------
best_model_path = Path(OUTPUT_DIR) / f"{MODEL_NAME}.keras"
callbacks = [
    ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

hist = pd.DataFrame(history.history)
hist["epoch"] = np.arange(1, len(hist) + 1)

# Aggregate numeric and categorical metrics to simplify reading.
num_mse_cols = [f"{head}_mse" for head in numeric_head_specs if f"{head}_mse" in hist.columns]
num_mae_cols = [f"{head}_mae" for head in numeric_head_specs if f"{head}_mae" in hist.columns]
val_num_mse_cols = [f"val_{head}_mse" for head in numeric_head_specs if f"val_{head}_mse" in hist.columns]
val_num_mae_cols = [f"val_{head}_mae" for head in numeric_head_specs if f"val_{head}_mae" in hist.columns]
if num_mse_cols:
    hist["numeric_group_mse_mean"] = hist[num_mse_cols].mean(axis=1)
if num_mae_cols:
    hist["numeric_group_mae_mean"] = hist[num_mae_cols].mean(axis=1)
if val_num_mse_cols:
    hist["val_numeric_group_mse_mean"] = hist[val_num_mse_cols].mean(axis=1)
if val_num_mae_cols:
    hist["val_numeric_group_mae_mean"] = hist[val_num_mae_cols].mean(axis=1)

train_cat_acc_cols = [
    c for c in hist.columns if c.startswith("cat__") and c.endswith("sparse_categorical_accuracy")
]
val_cat_acc_cols = [
    c for c in hist.columns if c.startswith("val_cat__") and c.endswith("sparse_categorical_accuracy")
]
train_cat_loss_cols = [c for c in hist.columns if c.startswith("cat__") and c.endswith("_loss")]
val_cat_loss_cols = [c for c in hist.columns if c.startswith("val_cat__") and c.endswith("_loss")]

if train_cat_acc_cols:
    hist["cat_acc_mean"] = hist[train_cat_acc_cols].mean(axis=1)
if val_cat_acc_cols:
    hist["val_cat_acc_mean"] = hist[val_cat_acc_cols].mean(axis=1)
if train_cat_loss_cols:
    hist["cat_loss_mean"] = hist[train_cat_loss_cols].mean(axis=1)
if val_cat_loss_cols:
    hist["val_cat_loss_mean"] = hist[val_cat_loss_cols].mean(axis=1)

hist.to_csv(Path(OUTPUT_DIR) / "train_history.csv", index=False)

plot_metric(hist, "loss", "val_loss", "Loss", "train_history_loss.png")
plot_metric(
    hist,
    "numeric_group_mse_mean",
    "val_numeric_group_mse_mean",
    "Numeric Group MSE",
    "train_history_numeric_group_mse.png",
)
plot_metric(
    hist,
    "numeric_group_mae_mean",
    "val_numeric_group_mae_mean",
    "Numeric Group MAE",
    "train_history_numeric_group_mae.png",
)
plot_metric(
    hist,
    "freq_head_mse",
    "val_freq_head_mse",
    "Freq Head MSE",
    "train_history_freq_mse.png",
)
plot_metric(
    hist,
    "cat_acc_mean",
    "val_cat_acc_mean",
    "Categorical Accuracy (mean)",
    "train_history_cat_acc_mean.png",
)
plot_metric(
    hist,
    "cat_loss_mean",
    "val_cat_loss_mean",
    "Categorical Loss (mean)",
    "train_history_cat_loss_mean.png",
)


# -----------------------------------------------------------------------------
# Prediction helpers
# -----------------------------------------------------------------------------
def inverse_numeric_head(head_name: str, values_norm: np.ndarray) -> np.ndarray:
    scaler = scaler_by_numeric_head[head_name]
    values = scaler.inverse_transform(values_norm.astype(np.float32))
    spec = numeric_head_specs[head_name]
    if spec["group_name"] == "ratio":
        values = np.power(2.0, values)
    return values


def inverse_freq_head(values_norm: np.ndarray) -> np.ndarray:
    if scaler_freq is None:
        raise RuntimeError("freq_head scaler is not available.")
    values_log2 = scaler_freq.inverse_transform(values_norm.astype(np.float32))
    return np.power(2.0, values_log2)


# -----------------------------------------------------------------------------
# Test set evaluation
# -----------------------------------------------------------------------------
pred_raw = model.predict(predict_ds, verbose=1)

if isinstance(pred_raw, dict):
    pred_map = pred_raw
elif isinstance(pred_raw, list):
    pred_map = {name: pred for name, pred in zip(model.output_names, pred_raw)}
else:
    pred_map = {model.output_names[0]: pred_raw}

pred_model = pd.DataFrame(index=test_raw.index)

for head_name, spec in numeric_head_specs.items():
    pred_values = inverse_numeric_head(head_name, np.asarray(pred_map[head_name]))
    pred_model[spec["cols"]] = pred_values

if freq_col is not None:
    pred_freq = inverse_freq_head(np.asarray(pred_map["freq_head"]).reshape(-1, 1))[:, 0]
    pred_model[freq_col] = pred_freq

for col in categorical_cols:
    head_name = cat_output_name(col)
    pred_logits = np.asarray(pred_map[head_name])
    pred_cls = np.argmax(pred_logits, axis=1).astype(np.int32)
    pred_model[col] = pred_cls

pred_model = pred_model[target_model.columns]

pred_full = pred_model.copy()
for col, value in constant_targets.items():
    pred_full[col] = value

pred_full = pred_full[target_all.columns]
y_true_full = test_all[target_all.columns]

y_true_np = y_true_full.to_numpy(dtype=np.float32)
y_pred_np = pred_full.to_numpy(dtype=np.float32)

mse = float(np.mean((y_true_np - y_pred_np) ** 2))
mae = float(np.mean(np.abs(y_true_np - y_pred_np)))
rmse = float(np.sqrt(mse))

print(f"RMSE Test: {rmse}")
print(f"MSE Test: {mse}")
print(f"MAE Test: {mae}")

metrics_extra = {}
test_metrics_by_head = {}

# Frequency metrics.
if freq_col is not None:
    freq_true = y_true_full[freq_col].to_numpy(dtype=np.float64)
    freq_pred = pred_full[freq_col].to_numpy(dtype=np.float64)
    abs_err_hz = np.abs(freq_pred - freq_true)
    rel_err = abs_err_hz / np.maximum(freq_true, 1e-9)
    cents_err = np.abs(
        1200.0 * np.log2(np.clip(freq_pred, 1e-3, None) / np.clip(freq_true, 1e-3, None))
    )

    metrics_extra.update(
        {
            "freq_mae_hz": float(abs_err_hz.mean()),
            "freq_rmse_hz": float(np.sqrt(np.mean((freq_pred - freq_true) ** 2))),
            "freq_mape": float(rel_err.mean()),
            "freq_mae_cents": float(cents_err.mean()),
        }
    )
    test_metrics_by_head["freq_head"] = {
        "mae_hz": float(abs_err_hz.mean()),
        "rmse_hz": float(np.sqrt(np.mean((freq_pred - freq_true) ** 2))),
        "mse_hz": float(np.mean((freq_pred - freq_true) ** 2)),
        "mape": float(rel_err.mean()),
        "mae_cents": float(cents_err.mean()),
    }

# Numeric group metrics.
for head_name, spec in numeric_head_specs.items():
    cols = spec["cols"]
    true_group = y_true_full[cols].to_numpy(dtype=np.float64)
    pred_group = pred_full[cols].to_numpy(dtype=np.float64)
    group_err = pred_group - true_group
    test_metrics_by_head[head_name] = {
        "group_name": spec["group_name"],
        "mse_mean": float(np.mean(group_err**2)),
        "mae_mean": float(np.mean(np.abs(group_err))),
        "mse_by_col": {col: float(val) for col, val in zip(cols, np.mean(group_err**2, axis=0), strict=False)},
        "mae_by_col": {col: float(val) for col, val in zip(cols, np.mean(np.abs(group_err), axis=0), strict=False)},
    }

# Categorical metrics.
categorical_accuracy = {}
categorical_crossentropy = {}
for col in categorical_cols:
    true_codes = y_true_full[col].to_numpy(dtype=np.int32)
    pred_codes = pred_full[col].to_numpy(dtype=np.int32)
    categorical_accuracy[col] = float(np.mean(true_codes == pred_codes))

    head_name = cat_output_name(col)
    y_pred_prob = np.asarray(pred_map[head_name], dtype=np.float64)
    ce = tf.keras.losses.sparse_categorical_crossentropy(true_codes, y_pred_prob).numpy()
    categorical_crossentropy[col] = float(np.mean(ce))

if categorical_accuracy:
    metrics_extra["categorical_accuracy_mean"] = float(np.mean(list(categorical_accuracy.values())))
    test_metrics_by_head["categorical_heads"] = {
        "accuracy_mean": float(np.mean(list(categorical_accuracy.values()))),
        "crossentropy_mean": float(np.mean(list(categorical_crossentropy.values()))),
        "accuracy_by_col": categorical_accuracy,
        "crossentropy_by_col": categorical_crossentropy,
    }

# Group summaries to mirror training history.
if numeric_head_specs:
    group_mse_vals = [
        test_metrics_by_head[head_name]["mse_mean"]
        for head_name in numeric_head_specs
        if head_name in test_metrics_by_head
    ]
    group_mae_vals = [
        test_metrics_by_head[head_name]["mae_mean"]
        for head_name in numeric_head_specs
        if head_name in test_metrics_by_head
    ]
    metrics_extra["numeric_group_mse_mean"] = float(np.mean(group_mse_vals))
    metrics_extra["numeric_group_mae_mean"] = float(np.mean(group_mae_vals))

print("===== Test Metrics by Head =====")
for head_name in numeric_head_specs:
    print(
        f"{head_name}: mse_mean={test_metrics_by_head[head_name]['mse_mean']:.6f} "
        f"mae_mean={test_metrics_by_head[head_name]['mae_mean']:.6f}"
    )
if "freq_head" in test_metrics_by_head:
    print(
        "freq_head: "
        f"mae_hz={test_metrics_by_head['freq_head']['mae_hz']:.3f} "
        f"rmse_hz={test_metrics_by_head['freq_head']['rmse_hz']:.3f} "
        f"mape={test_metrics_by_head['freq_head']['mape']:.6f} "
        f"mae_cents={test_metrics_by_head['freq_head']['mae_cents']:.3f}"
    )
if "categorical_heads" in test_metrics_by_head:
    print(
        "categorical_heads: "
        f"accuracy_mean={test_metrics_by_head['categorical_heads']['accuracy_mean']:.6f} "
        f"crossentropy_mean={test_metrics_by_head['categorical_heads']['crossentropy_mean']:.6f}"
    )


# -----------------------------------------------------------------------------
# Save artifacts
# -----------------------------------------------------------------------------
model.save(best_model_path)

preprocess_bundle = {
    "target_columns": target_columns,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "freq_col": freq_col,
    "constant_targets": constant_targets,
    "categorical_maps": categorical_maps,
    "numeric_head_specs": numeric_head_specs,
    "scaler_by_numeric_head": scaler_by_numeric_head,
    "scaler_freq": scaler_freq,
    "sample_rate_out": sample_rate_out,
    "duration_seconds": duration_seconds,
    "target_samples": target_samples,
}
joblib.dump(preprocess_bundle, Path(OUTPUT_DIR) / f"target_preprocess_{MODEL_NAME}.save")

# Save predictions for the test split.
pd.DataFrame(pred_model, columns=target_model.columns).to_csv(
    Path(OUTPUT_DIR) / "params_pred_test.csv",
    index=False,
)

pred_export = {}
for col in target_model.columns:
    pred_export[f"true_{col}"] = y_true_full[col].to_numpy()
    pred_export[f"pred_{col}"] = pred_model[col].to_numpy()
pd.DataFrame(pred_export).to_csv(Path(OUTPUT_DIR) / "predictions_test.csv", index=False)

pred_full.to_csv(Path(OUTPUT_DIR) / "params_pred_test_full.csv", index=False)

# Results JSON.
history_last = {}
for key in hist.columns:
    if key == "epoch":
        continue
    history_last[key] = to_json_scalar(hist[key].iloc[-1])

best_val_loss_epoch = None
best_val_loss = None
if "val_loss" in hist:
    best_idx = int(hist["val_loss"].idxmin())
    best_val_loss_epoch = int(hist.loc[best_idx, "epoch"])
    best_val_loss = float(hist.loc[best_idx, "val_loss"])

results = {
    "model_name": MODEL_NAME,
    "dataset": BASE_PATH,
    "output_dir": OUTPUT_DIR,
    "tamanho_dataset": int(tamanho_dataset),
    "split": {
        "train_size": int(len(train_idx)),
        "val_size": int(len(val_local_idx)),
        "test_size": int(len(test_idx)),
        "train_frac": float(TRAIN_FRAC),
        "val_frac_of_train": float(VAL_FRAC_OF_TRAIN),
    },
    "runtime_config": {
        "batch_size": int(BATCH_SIZE),
        "pred_batch_size": int(PRED_BATCH_SIZE),
        "epochs": int(EPOCHS),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "mixed_precision": bool(gpus),
        "policy": mixed_precision.global_policy().name,
        "target_samples": int(target_samples),
    },
    "model": {
        "num_params": int(model.count_params()),
    },
    "target_preprocessing": {
        "target_columns": target_columns,
        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "freq_col": freq_col,
        "constant_targets": constant_targets,
        "categorical_maps": categorical_maps,
        "numeric_head_specs": numeric_head_specs,
    },
    "metrics": {
        "mse": float(mse),
        "mae": float(mae),
        "rmse": float(rmse),
    },
    "metrics_extra": metrics_extra,
    "categorical_accuracy": categorical_accuracy,
    "categorical_crossentropy": categorical_crossentropy,
    "test_metrics_by_head": test_metrics_by_head,
    "history_last": history_last,
    "best_val_loss": {
        "epoch": best_val_loss_epoch,
        "value": best_val_loss,
    },
    "categorical_maps": categorical_maps,
}

with open(Path(OUTPUT_DIR) / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved results to {Path(OUTPUT_DIR) / 'results.json'}")
